In [1]:
import json
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import mols2grid

import warnings
warnings.filterwarnings("ignore", category=UserWarning)



pdb_directory = Path('/mnt/ligandpro/db/PDB/pdb2/processed')



In [2]:
def extract_hetatm_info(pdb_file):
    hetatm_types = []
    with open(pdb_file, 'r') as file:
        for line in file:
            if line.startswith("HETATM"):
                hetatm_type = line[17:20].strip()
                hetatm_types.append(hetatm_type)
    return hetatm_types

def process_pdb_files(pdb_directory):
    pdb_files = list(pdb_directory.glob('*.pdb'))
    hetatm_counts = Parallel(n_jobs=-1)(delayed(extract_hetatm_info)(pdb_file) for pdb_file in tqdm(pdb_files))
    hetatm_types = [item for sublist in hetatm_counts for item in sublist]
    return pd.Series(hetatm_types).value_counts()

hetatm_distribution = process_pdb_files(pdb_directory)

hetatm_dict = hetatm_distribution.to_dict()

json_path = Path('../data/hetatm_distribution.json')
with open(json_path, 'w') as json_file:
    json.dump(hetatm_dict, json_file)


hetatm_json_str = json.dumps(hetatm_dict, indent=4)
print(hetatm_json_str)


 77%|███████▋  | 167601/218077 [00:10<00:02, 16854.12it/s]

In [ ]:
hetatm_keys = list(hetatm_dict.keys())
print(hetatm_keys)

['NAG', 'CLA', 'MSE', 'PX4', 'HEM', 'GOL', 'SO4', 'PCW', 'EDO', 'DPV', 'FAD', 'HEC', 'ADP', 'NAP', 'MAN', 'NAD', 'ATP', '7Q9', 'BCR', 'CDL', 'POV', '17F', 'BMA', 'BCL', 'MG', 'NDP', 'LMT', 'FMN', 'GTP', 'PEG', 'PO4', 'GDP', 'DMS', 'GLC', 'CHL', 'LHG', 'ZN', 'COA', '3PE', 'CLR', 'MLY', 'MC3', 'ACT', 'LMG', 'BGC', 'ANP', 'DOD', 'HYP', 'Y01', 'SAH', 'MPD', 'OLC', 'DGD', 'CYC', 'SEP', 'CA', 'PG4', 'AGS', 'PEE', 'NAI', 'CL', 'PC1', 'UNL', 'FUC', 'SF4', 'AMP', 'FOL', 'GAL', 'MES', 'PGE', 'PTR', 'GNP', 'TPO', 'MXT', 'PLP', 'SQD', 'CIT', '1PE', 'EPE', 'PLM', 'NH2', 'PTY', 'PLC', 'ACE', 'LLP', 'LCG', 'OLA', 'NA', 'PGV', 'ACO', 'MYR', 'SAM', 'PSU', 'BOG', 'FMT', 'TRS', 'UDP', '5CM', 'LUT', 'A2G', 'CGU', 'B12', 'PHO', 'LDA', 'HEA', 'LFA', 'SIA', 'DAL', 'DMU', 'DPR', 'PEB', 'PCA', 'GSH', 'PLX', 'MTN', 'ALY', 'KCX', 'AIB', 'C8E', 'PL9', 'CME', 'TGL', 'DLE', 'H4B', 'C14', 'SGN', 'RET', 'CHD', 'FME', 'ACP', 'BPH', '8OG', 'TPP', 'C2E', 'M3L', 'DLY', 'IDS', 'PEK', 'MLE', 'P6G', 'MN', 'II0', 'PGW', 'RCY

In [ ]:
import os
from Bio import PDB

def parse_pdb_file(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('', file_path)

    has_protein = False
    ligands = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if PDB.is_aa(residue, standard=True):
                    has_protein = True

                elif residue.id[0] != " ":
                    ligand_id = residue.resname
                    res_number = residue.id[1]
                    chain_id = chain.id
                    ligands.append((ligand_id, res_number, chain_id))

    if has_protein and ligands:
        return ligands
    else:
        return None

def extract_complexes_with_ligands(pdb_directory, max_files=1):
    pdb_complexes = {}
    processed_files = 0
    
    for pdb_file in os.listdir(pdb_directory):
        if pdb_file.endswith(".pdb"):
            # Извлекаем pdb_id, начиная с 4-го символа
            pdb_id = pdb_file[3:].split('.')[0]
            file_path = os.path.join(pdb_directory, pdb_file)
            ligands = parse_pdb_file(file_path)

            if ligands:
                pdb_complexes[pdb_id] = ligands
            
            processed_files += 1
            if processed_files >= max_files:
                break
    
    return pdb_complexes

if __name__ == "__main__":
    pdb_directory = "/mnt/ligandpro/db/PDB/pdb2/processed" 
    complexes = extract_complexes_with_ligands(pdb_directory, max_files=10)
    print(complexes)


{'7dvc': [('CL', 101, 'A'), ('CL', 102, 'A'), ('CL', 103, 'A'), ('ACT', 101, 'C'), ('CL', 102, 'C'), ('CL', 103, 'C'), ('CL', 104, 'C'), ('ACT', 101, 'E'), ('CL', 102, 'E'), ('ACT', 101, 'F'), ('ACT', 101, 'B'), ('CL', 102, 'B'), ('ACT', 101, 'D'), ('CL', 102, 'D')], '2xb0': [('GOL', 1266, 'X'), ('CL', 1267, 'X'), ('CL', 1268, 'X'), ('GOL', 1269, 'X')], '6qxw': [('NA', 201, 'A'), ('NA', 202, 'A'), ('CL', 203, 'A'), ('CL', 204, 'A'), ('EDO', 205, 'A')], '2mfq': [('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B'), ('PTR', 512, 'B')], '8p3w': [('PLM', 1001, 'A'), ('OLC', 1002, 'A'), ('PLM', 1001, 'B'), ('OLC', 1002, 'B'), ('POV', 1003, 'B'), ('PLM', 1001, 'C'), ('OLC', 1002, 'C'), ('PL

In [ ]:
complexes

{'7dvc': [('CL', 101, 'A'),
  ('CL', 102, 'A'),
  ('CL', 103, 'A'),
  ('ACT', 101, 'C'),
  ('CL', 102, 'C'),
  ('CL', 103, 'C'),
  ('CL', 104, 'C'),
  ('ACT', 101, 'E'),
  ('CL', 102, 'E'),
  ('ACT', 101, 'F'),
  ('ACT', 101, 'B'),
  ('CL', 102, 'B'),
  ('ACT', 101, 'D'),
  ('CL', 102, 'D')],
 '2xb0': [('GOL', 1266, 'X'),
  ('CL', 1267, 'X'),
  ('CL', 1268, 'X'),
  ('GOL', 1269, 'X')],
 '6qxw': [('NA', 201, 'A'),
  ('NA', 202, 'A'),
  ('CL', 203, 'A'),
  ('CL', 204, 'A'),
  ('EDO', 205, 'A')],
 '2mfq': [('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B'),
  ('PTR', 512, 'B')],
 '8p3w': [('PLM', 1001, 'A'),
  ('OLC', 1002, 'A'),
  ('PLM', 1001, 'B')